# 1: Pre-process Documents

In [1]:
from unstructured.partition.pdf import partition_pdf
from unstructured.cleaners.core import (
    clean,
    clean_extra_whitespace,
    clean_bullets,
    clean_ordered_bullets,
    clean_dashes,
    clean_trailing_punctuation,
    replace_unicode_quotes,
)
from unstructured.documents.elements import (
    Title,
    NarrativeText,
    ListItem,
    Table,
)
from unstructured.chunking.title import chunk_by_title

In [2]:
ALLOWED_TYPES = (Title, NarrativeText, ListItem, Table)

MIN_TEXT_LENGTH = 20
MAX_CHARACTERS = 1000
NEW_AFTER_N_CHARS = 800
COMBINE_TEXT_UNDER_N_CHARS = 200

## 1.1: Document Parsing

In [3]:
def parse_pdf_document(file_path: str) -> list:
    elements = partition_pdf(
        filename=file_path,
        strategy="hi_res",
        infer_table_structure=True,
        include_page_breaks=True,
        extract_images_in_pdf=False,
        languages=["eng"],
    )
    return elements

## 1.2: Element Cleaning

In [4]:
def clean_element_text(text: str) -> str:
    if not text:
        return ""

    text = replace_unicode_quotes(text)
    text = clean_bullets(text)
    text = clean_ordered_bullets(text)
    text = clean_dashes(text)
    text = clean_trailing_punctuation(text)
    text = clean_extra_whitespace(text)

    text = clean(
        text,
        extra_whitespace=True,
        dashes=False,
        bullets=False,
        trailing_punctuation=False,
        lowercase=False,
    )

    return text.strip()

In [5]:
def clean_elements(elements):
    cleaned = []

    for el in elements:
        if not getattr(el, "text", None):
            continue

        text = clean_element_text(el.text)

        if not text:
            continue

        el.text = text
        cleaned.append(el)

    return cleaned

## 1.3: Filter Elements

In [6]:
def filter_elements(elements):
    filtered = []

    for el in elements:
        if not isinstance(el, ALLOWED_TYPES):
            continue

        if not el.text:
            continue

        if len(el.text.strip()) < MIN_TEXT_LENGTH:
            continue

        filtered.append(el)

    return filtered

## 1.4: Element Chunking

In [7]:
def chunk_elements_by_title(elements):
    chunks = chunk_by_title(
        elements,
        max_characters=MAX_CHARACTERS,
        new_after_n_chars=NEW_AFTER_N_CHARS,
        combine_text_under_n_chars=COMBINE_TEXT_UNDER_N_CHARS,
        include_orig_elements=True,
    )

    return chunks

## 1.5: PDF Pre-processing Pipeline

In [8]:
def run_pipeline(file_path: str):
    print(f"[1/4] Parsing: {file_path}")
    elements = parse_pdf_document(file_path)
    print(f"       → {len(elements)} elements")

    print("[2/4] Cleaning")
    elements = clean_elements(elements)
    print(f"       → {len(elements)} elements")

    print("[3/4] Filtering")
    elements = filter_elements(elements)
    print(f"       → {len(elements)} elements")

    print("[4/4] Chunking")
    chunks = chunk_elements_by_title(elements)
    print(f"       → {len(chunks)} chunks")

    return chunks

# 1.6: Start Pre-processing PDF Documents

In [10]:
file_path = "data/pdfs/LLMs.pdf"
chunks = run_pipeline(file_path)

[1/4] Parsing: data/pdfs/LLMs.pdf
       → 790 elements
[2/4] Cleaning
       → 680 elements
[3/4] Filtering
       → 167 elements
[4/4] Chunking
       → 99 chunks


In [ ]:
chunks[0].to_dict()

{'type': 'CompositeElement',
 'element_id': '1f3d12e0-2a7d-4626-9669-9aa69a983b1f',
 'text': 'Can Large Language Models Replace Data Scientists in Clinical Research? Zifeng Wang1∗, Benjamin Danek1∗, Ziwei Yang3, Zheng Chen4, Jimeng Sun1,2# 1 Department of Computer Science, University of Illinois Urbana Champaign, Champaign, IL\n\n2 Carle Illinois College of Medicine, University of Illinois Urbana Champaign, Champaign, IL 3 Bioinformatics Center, Institute for Chemical Research, Kyoto University',
 'metadata': {'file_directory': 'data/pdfs',
  'filename': 'LLMs.pdf',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'last_modified': '2026-03-13T21:52:30',
  'page_number': 1,
  'orig_elements': 'eJzlU01v2zAM/SuCd3U9S/JnLwOaXrqlw7C2GNasMGSbSrTZkiEpbYNi/32Uk27dUOxQYKcBgRE+PlLkI7l6iGCAEbRvVB8dkwh4IXndZywt2rQsU86A521JAWgNZVZGMYlG8KIXXiD/IeqMsb3SwoOb7UHszNY3G1DrjUeEV3mKMQf4TvV+gyiryxzRySjtQ9xqxes84TFhVZHQm5g82jkvkjLYLKdFUj0D7CMQidzOeRhDFx/UPQwXk+gg+o6OHjx0XhnddINwrpmsaZGWJlXBMEMk1QBN

In [ ]:
for i, chunk in enumerate(chunks):
        print(f"[{i}] {len(chunk.text)} chars")
        print(chunk.text[:200])
        print("-" * 80)

[0] 403 chars
Can Large Language Models Replace Data Scientists in Clinical Research? Zifeng Wang1∗, Benjamin Danek1∗, Ziwei Yang3, Zheng Chen4, Jimeng Sun1,2# 1 Department of Computer Science, University of Illino
--------------------------------------------------------------------------------
[1] 990 chars
Data science plays a critical role in clinical research, but it requires professionals with expertise in coding and medical data analysis. Large language models (LLMs) have shown great potential in su
--------------------------------------------------------------------------------
[2] 863 chars
adaptation methods and found two to be particularly effective: chain of thought prompting, which provides a step by step plan for data analysis, which led to a 60% im provement in code accuracy; and s
--------------------------------------------------------------------------------
[3] 997 chars
In clinical research, data science plays a pivotal role in analyzing complex datasets, such as cli

# 2: Store Chunks to Vector Database